# PSM prod-extraction-v1 — Colab smoke train

Phase 5 production-memory training on **Hugging Face + Colab** (not RunPod).

- Resume **`058000`** from [`subbu83/psm-50m-mixed-v1-run`](https://huggingface.co/subbu83/psm-50m-mixed-v1-run)
- Curriculum **`prod-extraction-v1`** from [`chkrishna2001/psm-50m-action-mixed-v1`](https://huggingface.co/datasets/chkrishna2001/psm-50m-action-mixed-v1)
- Smoke: **+2000 steps** (058000 → 078000), LR **2e-5**, prod grounding eval every **500** steps

Runtime: **GPU** (T4/L4/A100).

**HF tokens** — cell **1b** prompts if Colab secrets are missing:
- **Model** `HF_TOKEN` — read on `subbu83/psm-50m-mixed-v1-run` (write too if uploading checkpoints)
- **Dataset** `DATASET_HF_TOKEN` — read on `chkrishna2001/psm-50m-action-mixed-v1` (can be same token)

Create at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Optional: Colab **Secrets** (🔑 sidebar) → `HF_TOKEN`, `DATASET_HF_TOKEN`.

## 1. Setup

In [ ]:
!pip install -q huggingface_hub hf_transfer numpy
![ -d /content/PSM ] || git clone --depth 1 https://github.com/chkrishna2001/PSM.git /content/PSM
%cd /content/PSM

In [ ]:
import os
from getpass import getpass

def _colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def _token(name: str, *, prompt: str, optional: bool = False) -> str:
    value = os.environ.get(name) or _colab_secret(name)
    if value:
        return value.strip()
    if optional:
        return ""
    print(prompt)
    print("(input hidden — paste token and press Enter)")
    entered = getpass(f"{name}: ").strip()
    if not entered:
        raise ValueError(f"{name} is required. Set a Colab secret or paste at the prompt.")
    return entered

os.environ["HF_TOKEN"] = _token(
    "HF_TOKEN",
    prompt="Model repo token (subbu83/psm-50m-mixed-v1-run): needs read; write if uploading steps.",
)
dataset_token = _token(
    "DATASET_HF_TOKEN",
    prompt="Dataset repo token (chkrishna2001/psm-50m-action-mixed-v1). Press Enter to reuse HF_TOKEN.",
    optional=True,
)
os.environ["DATASET_HF_TOKEN"] = dataset_token or os.environ["HF_TOKEN"]
os.environ["PYTHONPATH"] = "psm-model/src:psm-model/prod-memory"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))
print("DATASET_HF_TOKEN set:", bool(os.environ["DATASET_HF_TOKEN"]))

## 2. CUDA check

In [ ]:
import json, torch
print(json.dumps({
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
assert torch.cuda.is_available(), "Switch runtime to GPU."

## 3. Download HF assets (058000 + curriculum + probes)

In [ ]:
!PYTHONPATH=psm-model/src:psm-model/prod-memory python -m prod_memory.colab_sync download --root /content/PSM

In [ ]:
from pathlib import Path
candidates = {
    "checkpoint": ["psm-model/checkpoints/real-v3-50m-full-v2-step-058000.pt"],
    "curriculum": ["prod-memory/prod-extraction-v1.jsonl"],
    "expanded_probe": [
        "data/direct-behavior-v1/expanded-probe-v1-filtered.jsonl",
        "probes/expanded-probe-v1-filtered.jsonl",
    ],
    "direct_probe": ["data/probes/direct_probes.jsonl", "probes/direct_probes.jsonl"],
}
resolved = {}
missing = []
for key, paths in candidates.items():
    hit = next((p for p in paths if Path(p).exists()), None)
    if hit:
        resolved[key] = hit
    else:
        missing.append(key)
print("resolved:", resolved)
print("missing:", missing)
assert not missing

## 4. Train (+2000 steps from 058000)

In [ ]:
RESUME = "psm-model/checkpoints/real-v3-50m-full-v2-step-058000.pt"
TOK = "psm-model/checkpoints/real-v3-50m-full-v2-step-058000.tokenizer.json"
CURRICULUM = "prod-memory/prod-extraction-v1.jsonl"
TARGET_STEPS = 78000
BATCH_SIZE = 8  # use 4 if OOM
LEARNING_RATE = 2e-5

!PYTHONPATH=psm-model/src python -m psm_model.train \
  {CURRICULUM} \
  --out psm-model/checkpoints/real-v3-50m-full-v2.pt \
  --resume {RESUME} \
  --tokenizer {TOK} \
  --steps {TARGET_STEPS} \
  --batch-size {BATCH_SIZE} \
  --learning-rate {LEARNING_RATE} \
  --min-learning-rate 5e-6 \
  --warmup-steps 50 \
  --preset 50m \
  --output-format tagged \
  --device cuda \
  --save-every 500 \
  --metrics-out psm-model/checkpoints/real-v3-50m-full-v2-prod-extraction.metrics.jsonl \
  --structural-loss-weight 1 \
  --eval-every 500 \
  --probe {resolved['expanded_probe']} \
  --manual-probe {resolved['direct_probe']} \
  --abort-after-step 90000 \
  --collapse-threshold 0.90

## 5. Prod grounding eval (Phase 1 fixtures)

In [ ]:
import json
from pathlib import Path

ckpt_dir = Path("psm-model/checkpoints")
eval_steps = sorted(int(p.stem.split("-")[-1]) for p in ckpt_dir.glob("real-v3-50m-full-v2-step-*.pt"))
print("checkpoints:", eval_steps[-5:])

for step in eval_steps:
    if step < 58500 or step > TARGET_STEPS:
        continue
    ckpt = ckpt_dir / f"real-v3-50m-full-v2-step-{step:06d}.pt"
    out = Path(f"psm-model/prod-memory/results/prod-grounding-step-{step:06d}.json")
    out.parent.mkdir(parents=True, exist_ok=True)
    !PYTHONPATH=psm-model/src:psm-model/prod-memory python -m prod_memory.eval_grounding \
      --checkpoint {ckpt} \
      --checkpoint-label {step} \
      --device cuda \
      --out {out}

## 6. Upload step checkpoints to HF (optional)

In [ ]:
UPLOAD_STEPS = [s for s in eval_steps if s > 58000]
print("upload steps:", UPLOAD_STEPS)
!PYTHONPATH=psm-model/src:psm-model/prod-memory python -m prod_memory.colab_sync upload-steps \
  --root /content/PSM \
  --steps {' '.join(str(s) for s in UPLOAD_STEPS)}

## 7. Download results locally

From Colab: **Files → download** `psm-model/prod-memory/results/prod-grounding-step-*.json` and metrics JSONL.